# Synthea Explorations

This notebook explores synthetic patient data generated by [Synthea](https://github.com/synthetichealth/synthea).
Synthea outputs FHIR R4 Bundles containing Patient, Practitioner, Encounter, Condition, and other resources.

In [10]:
import os

SYNTHEA_OUTPUT_FOLDER = "../../../synthea/output/fhir"
MAX_BUNDLES = 250


# Check if the output folder exists
if not os.path.exists(SYNTHEA_OUTPUT_FOLDER):
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' does not exist.")
else:
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' found.")

Output folder '../../../synthea/output/fhir' found.


In [ ]:
from fhir.resources.R4B.patient import Patient

from dicom_py_mock_server.services.synthea_utils import SyntheaUtils

# Load the Synthea output data, parse them into a list of FHIR Bundle dicts, and return the list

syn = SyntheaUtils(synthea_output_folder=SYNTHEA_OUTPUT_FOLDER)

bundles = syn.scan_synthea_output(modalities=["CT", "US", "CR"], max_count=MAX_BUNDLES)

print(f"Loaded {len(bundles)} FHIR Bundles from Synthea output.")

for bundle in bundles:
    patient: Patient = syn.get_patient(bundle)
    dicom_name = syn.get_patient_name(patient)
    patient_id = syn.get_patient_id(patient)
    sex = syn.get_patient_sex(patient)
    birth_date = syn.get_date_time(patient.birthDate)
    # Encounter
    encounters = syn.get_encounters(bundle)
    diags = syn.get_diagnostic_reports(bundle)
    # Imaging Studies
    imagings = syn.get_imaging_studies(bundle)
    for imaging in imagings:
        for s in imaging.series:
            modality = s.modality.display
            print(f"Modality: {modality}")
        enc_ref = imaging.encounter.reference.replace("urn:uuid:", "")
        for enc in encounters:
            if enc_ref == enc.id:
                print(f"{enc.period.start}")
        for diag in diags:
            diag_enc = diag.encounter.reference.replace("urn:uuid:", "")
            if diag_enc == enc_ref:
                if diag and diag.presentedForm:
                    for form in diag.presentedForm:
                        match form:
                            case Attachment as a:
                                print(f"{diag_enc}: {a.data}")
    print(f"Patient: DICOM Name: {dicom_name}, Gender: {sex}, ID: {patient_id}, Birth Date: {birth_date[0]}")